# Acoustic 项目独立研究审计（可复现）

**文档性质**：独立、怀疑主义、以可复现性为核心的研究审计。本 notebook 的所有 headline 数字都由下方代码单元在执行时**从 `result/` 一手文件重算**，而非从报告或对话摘要转写。

**审计日期**：2026-07-23 · **审计范围**：ICBHI 强方法源对齐、SPRSound zero-target 迁移、frozen-encoder target-head 诊断、四数据集 curation、候选论文主线新颖性。

**证据优先级**：原始 predictions / metrics / receipts / code（一手）> 本地 `docs/` > Notion 会议 > 对话摘要。

**只读事实核验边界**：外部事实仅取自原论文 / 官方 repo / 官方数据集页面。凡无法端到端复跑处（encoder 权重级）标注为"代码/receipt 级已核验、未端到端复算"。

> 运行方式：`Kernel → Restart & Run All`。代码单元只读 `result/*.json` 与 `event_manifest.jsonl`，不需要 GPU、torch 或原始音频。

## 0. 复现性设置与一手文件加载

In [1]:
import json, math
from pathlib import Path
import numpy as np, pandas as pd

# notebook 位于 docs/audits/ -> 项目根为 parents[2]
ROOT = Path.cwd()
while not (ROOT / "result").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("project root:", ROOT)

def load(p):
    return json.loads((ROOT / p).read_text())

R = "result"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

project root: /Users/zilongzeng/Research/Acoustic


## 1. 源对齐核验 — Patch-Mix 作者 checkpoint 在 ICBHI 上

**待验证假设**：ICBHI author-checkpoint Score = 62.1708，作者公布 62.17。

In [2]:
m = load(f"{R}/icbhi_patchmix_author_eval/metrics.json")
score = m["metrics"]["icbhi_score"]
author = m["author_posted_metrics"]["icbhi_score"]
print(f"recomputed ICBHI Score      = {score:.6f}")
print(f"author-posted Score         = {author}")
print(f"absolute gap                = {abs(score-author):.6f}")
print(f"selection (claim limit)     = {m['selection']!r}")
print(f"status                      = {m['status']!r}")
assert abs(score - 62.1708) < 1e-3, "ICBHI author score drift"
print("\n[OK] Patch-Mix 作者 checkpoint 推理级精确对齐；但 checkpoint 由 official test 选出 -> 非 clean source anchor。")

recomputed ICBHI Score      = 62.170814
author-posted Score         = 62.17
absolute gap                = 0.000814
selection (claim limit)     = 'author checkpoint selected on official test Score'
status                      = 'completed_author_checkpoint_only_icbhi_test_selected'

[OK] Patch-Mix 作者 checkpoint 推理级精确对齐；但 checkpoint 由 official test 选出 -> 非 clean source anchor。


## 2. Zero-Target 迁移核验（真正的跨数据集测试）

源 ICBHI 分类头直接迁移到固定的 1,429 个 SPRSound inter events，无任何 target 训练/阈值/校准/选择。
**待验证假设**：binary Score Patch-Mix 59.38 / PAFA 55.82 / SG-SCL 59.98；narrow-four ≈ all-normal floor(50)。

In [3]:
def zt(path, kind):
    d = load(path)
    if kind == "b0":   # patchmix file layout
        return (d["b0"]["binary_broad"]["metrics"]["icbhi_score"],
                d["b0"]["narrow_four"]["metrics"]["icbhi_score"])
    return (d["transfer"]["binary_broad"]["icbhi_score"],
            d["transfer"]["narrow_four"]["icbhi_score"])

rows = []
for name, path, kind in [
    ("Patch-Mix", f"{R}/sprsound_patchmix_frozen_transfer/metrics.json", "b0"),
    ("PAFA",      f"{R}/pafa_sprsound_transfer_20260722_235659/metrics.json", "t"),
    ("SG-SCL",    f"{R}/sg_scl_sprsound_transfer_20260722_235659/metrics.json", "t"),
]:
    b, n = zt(path, kind)
    rows.append({"method": name, "binary_Score_(SE+SP)/2": round(b, 4),
                 "narrow4_Score": round(n, 4), "vs_floor(50)_pp": round(b-50, 2)})
zt_df = pd.DataFrame(rows); print(zt_df.to_string(index=False))
print("\n[OK] binary 仅高于 trivial floor 5.8–10.0 pp；narrow-four 贴 floor -> 源分类头直接迁移能力有限。")

   method  binary_Score_(SE+SP)/2  narrow4_Score  vs_floor(50)_pp
Patch-Mix                 59.3783        51.1521             9.38
     PAFA                 55.8209        51.1936             5.82
   SG-SCL                 59.9790        50.0818             9.98

[OK] binary 仅高于 trivial floor 5.8–10.0 pp；narrow-four 贴 floor -> 源分类头直接迁移能力有限。


## 3. Frozen-Encoder + Target-Head 核验（完整 target 监督下的探针）

冻结 ICBHI encoder、丢弃 source classifier/projector、在 SPRSound **全量 subtrain** 上训练随机初始化 `LayerNorm(768)+Linear` head（5 epochs，patient-grouped 内验证选 epoch，official inter 只评一次）。
**待验证假设**：binary Score 87.56/84.25/90.58；AS 87.68/84.29/90.65；seven-class Score 82.37/75.77/84.41；seven-class macro-F1 ≈ 0.362/0.387/0.372。

In [4]:
def fh(mid):
    d = load(f"{R}/sprsound_{mid}_frozen_encoder_target_heads/metrics.json")
    b, s = d["binary"], d["seven_class"]
    # 独立重算官方 Score = (AS + HS)/2，AS = (SE+SP)/2
    SE, SP = b["sensitivity_percent"], b["specificity_percent"]
    AS = (SE + SP) / 2
    HS = 2*SE*SP/(SE+SP)
    recomputed = (AS + HS)/2
    return {
        "method": mid,
        "binary_Score": round(b["official_sprsound_score_percent"], 4),
        "binary_Score_recomputed_(AS+HS)/2": round(recomputed, 4),
        "binary_AS": round(b["average_score_percent"], 4),
        "seven_Score": round(s["official_sprsound_score_percent"], 4),
        "seven_macroF1": round(s["macro_f1"], 4),
        "seven_UAR": round(s["uar"], 4),
    }
fh_df = pd.DataFrame([fh("patchmix"), fh("pafa"), fh("sg_scl")])
print(fh_df.to_string(index=False))
# 公式一致性断言
for _, r in fh_df.iterrows():
    assert abs(r["binary_Score"] - r["binary_Score_recomputed_(AS+HS)/2"]) < 1e-6
print("\n[OK] 官方 Score 公式 = (AS+HS)/2 独立重算一致（IEEE BioCAS 2022 定义）。")
print("[NB] seven-class Score 82–84 但 macro-F1 仅 0.36–0.39 -> 官方 Score 由事件流行度加权，掩盖细粒度失衡。")

  method  binary_Score  binary_Score_recomputed_(AS+HS)/2  binary_AS  seven_Score  seven_macroF1  seven_UAR
patchmix       87.5615                            87.5615    87.6824      82.3721         0.3618     0.3508
    pafa       84.2492                            84.2492    84.2875      75.7738         0.3871     0.2791
  sg_scl       90.5843                            90.5843    90.6495      84.4133         0.3720     0.3508

[OK] 官方 Score 公式 = (AS+HS)/2 独立重算一致（IEEE BioCAS 2022 定义）。
[NB] seven-class Score 82–84 但 macro-F1 仅 0.36–0.39 -> 官方 Score 由事件流行度加权，掩盖细粒度失衡。


## 4. Split / 患者不相交 / 类别构成 — 从 event_manifest 独立重算

**待验证假设**：subtrain 5219 / validation 1437 / inter 1429；subtrain∩validation 患者=0；train∩inter 患者=0；
inter support = Normal 1040 / Fine Crackle 80 / Coarse Crackle 3 / Wheeze 305 / Wheeze+Crackle 1 / Rhonchi 0 / Stridor 0。

In [5]:
from collections import Counter
manifest = [json.loads(l) for l in
            (ROOT/f"{R}/sprsound_patchmix_frozen_encoder_target_heads/event_manifest.jsonl").read_text().splitlines()]
train = [r for r in manifest if r["partition"]=="train"]
inter = [r for r in manifest if r["partition"]=="inter"]
sub  = [r for r in train if r["inner_split"]=="subtrain"]
val  = [r for r in train if r["inner_split"]=="validation"]
subP = {r["patient_id"] for r in sub}; valP = {r["patient_id"] for r in val}
trP  = {r["patient_id"] for r in train}; inP = {r["patient_id"] for r in inter}
print(f"subtrain events={len(sub)}  validation events={len(val)}  inter events={len(inter)}")
print(f"subtrain patients={len(subP)}  validation patients={len(valP)}  overlap={len(subP&valP)}")
print(f"train patients={len(trP)}  inter patients={len(inP)}  train∩inter={len(trP&inP)}")
assert (len(sub),len(val),len(inter))==(5219,1437,1429)
assert len(subP&valP)==0 and len(trP&inP)==0
# inter label-free 检查
print("inter manifest 含 raw_label 字段:", any("raw_label" in r for r in inter))
# inter support 从 scored predictions 独立统计
scored = pd.read_csv(ROOT/f"{R}/sprsound_patchmix_frozen_encoder_target_heads/inter_predictions_scored.csv")
print("\ninter raw_label support:", dict(sorted(Counter(scored['raw_label']).items())))
print("\n[OK] 患者严格不相交；inter manifest label-free；support 与假设逐项一致。")

subtrain events=5219  validation events=1437  inter events=1429
subtrain patients=194  validation patients=49  overlap=0
train patients=243  inter patients=41  train∩inter=0
inter manifest 含 raw_label 字段: False

inter raw_label support: {'Coarse Crackle': 3, 'Fine Crackle': 80, 'Normal': 1040, 'Wheeze': 305, 'Wheeze+Crackle': 1}

[OK] 患者严格不相交；inter manifest label-free；support 与假设逐项一致。


## 5. 度量定义不一致演示（M-1）

报告的"+28.18 pp 恢复"= frozen-head 官方 Score − zero-target Score，但两者公式不同：
zero-target 的 `icbhi_score` = **(SE+SP)/2 = AS**；frozen-head 的 `official_sprsound_score` = **(AS+HS)/2**。

In [6]:
b0 = load(f"{R}/sprsound_patchmix_frozen_transfer/metrics.json")["b0"]["binary_broad"]["metrics"]
fhb = load(f"{R}/sprsound_patchmix_frozen_encoder_target_heads/metrics.json")["binary"]
print(f"zero-target 'icbhi_score' = (SE+SP)/2      : {b0['icbhi_score']:.4f}")
print(f"frozen-head official Score = (AS+HS)/2     : {fhb['official_sprsound_score_percent']:.4f}")
print(f"frozen-head AS = (SE+SP)/2 (同 zero 口径)  : {fhb['average_score_percent']:.4f}")
print(f"报告差分 (混口径): {fhb['official_sprsound_score_percent']-b0['icbhi_score']:+.2f} pp")
print(f"同口径差分 AS-AS : {fhb['average_score_percent']-b0['icbhi_score']:+.2f} pp")
print("\n[FINDING M-1] 头条差分混用两种 Score 公式；binary 下差异小，但应统一口径后再相减。")

zero-target 'icbhi_score' = (SE+SP)/2      : 59.3783
frozen-head official Score = (AS+HS)/2     : 87.5615
frozen-head AS = (SE+SP)/2 (同 zero 口径)  : 87.6824
报告差分 (混口径): +28.18 pp
同口径差分 AS-AS : +28.30 pp

[FINDING M-1] 头条差分混用两种 Score 公式；binary 下差异小，但应统一口径后再相减。


---
# 审计结论（叙述部分）

以下为基于上方一手复算 + Notion 会议 + 一手论文的判断。数字均已在代码单元验证。

## Section 1 · Executive Verdict

**判定：研究流程在正确 track；实现严谨、数字精确、claim 边界克制——但（a）支撑"共享声学表示可跨数据集复用"的关键对照缺失，（b）候选论文主线相对 BTS-CARD / LungMix 新颖性不足。→「部分偏离」：过程无需重构，核心 claim 与论文主线需在做出方法承诺前先补诊断实验并重新定位。**

1. **过程正确、执行诚实**：Patch-Mix ICBHI Score 62.1708（gap 0.0008）；三个 frozen-head 的 binary/seven/AS/macro-F1 与报告及"待验证假设"逐位吻合（见 §1–§4）。encoder 冻结、source classifier/projector 丢弃、label 后加载、patient-disjoint 均为**硬 gate**（会抛异常）。
2. **claim 边界克制**：`degradation_claim_allowed_without_target_native_reference: false` 由代码强制；WORK_PLAN 禁止直接相减 source/target Score。
3. **两处结构性风险**：正向结果来自**完整 target 监督**（zero-target 仅比 floor 高 6–10 pp）；缺 AudioSet-only encoder 对照，无法证明 ICBHI 微调本身有贡献。

## Section 2 · Meeting-Alignment Matrix

| 导师指令（Notion） | 要求 | 状态 | 判定 |
|---|---|---|---|
| 2026-06-24 方向&Scope | 第一篇=Audio FM + cross-dataset + imbalance；LLM branch 推迟 | 全部工作在 audio branch，无 LLM 建模 | ✅ 对齐 |
| 2026-06-29 Jingping Record | 贡献落在 system/architecture/training，**非** dataset aggregation | 主线是模型/迁移诊断；候选主线含 label overlay 偏 dataset，需注意 | 🟡 张力 |
| 2026-06-29 | 设计新架构前先复现已有 imbalance 方法（tree/hierarchical） | 已复现强方法+长尾 loss；hierarchical/joint-head/fusion 已测且为**负结果** | ✅ 对齐 |
| 2026-06-29 | 评估协议 leave-one-dataset-out 等 | 仅 ICBHI→SPRSound 单向；LODO 未实现 | 🟡 部分 |
| 2026-07-13 Meeting Report | 复现强 ICBHI 方法 + 直接迁移 + 判断 degradation | 复现+迁移完成；degradation 正确地推迟到 C0 | ✅ 对齐 |
| 2026-07-23 审阅稿 | 四数据集声学分布分析（SNR/PSD/device/duration） | **本 notebook 配套的 acoustic_distribution 分析补上（见姊妹 notebook）** | 🟩 本次补齐 |
| 2026-07-23 | HF/KAUH adapter、半自动标注 pilot | 设计完成、执行未开始 | ❌ 未开始 |

## Section 3 · Result Verification Table

见 §1–§4 代码输出。全部 ✅ 精确吻合：ICBHI 62.1708；zero-target 59.38/55.82/59.98（narrow4≈floor）；frozen-head binary 87.56/84.25/90.58、AS 87.68/84.29/90.65、seven 82.37/75.77/84.41、macro-F1 0.362/0.387/0.372；split 5219/1437/1429、患者不相交、inter support 逐项一致。**未验证项**：三源 checkpoint 文件本身、PAFA/SG-SCL 源 ICBHI 数（服务器接收非本地重算）、SPRSound 77.42 原表精确上下文（经 DOI 二次确认）。

## Section 4 · Findings（Critical / High / Medium / Low）

**🔴 Critical**
- **C-1 缺 AudioSet-only encoder 对照**：三个 frozen-head 都用"ICBHI 微调后"的 AST/BEATs。没有"未微调的原始 AudioSet AST/BEATs + 同 head"对照，因此 87.56 的恢复可能主要来自 AudioSet 预训练 + 完整 target 监督，ICBHI 微调贡献未证。WORK_PLAN 已提及需 foundation-representation control 但未执行。→ 证据链最致命缺口。
- **C-2 正向结论建立在完整 target 监督上，而非跨数据集迁移**：5 epochs = 对全 5219 subtrain 的 5 次遍历（非 few-shot）；真正的 zero-target 只比 floor 高 6–10 pp。报告 §5 已诚实承认，但论文叙事有把"target-probing 强"误读为"跨数据集泛化强"的风险。

**🟠 High**
- **H-1 官方 Score 高是任务/度量结构使然**：binary(87–90) 与 ICBHI 四分类(62) 是不同 task/label-space/unit，**禁止相减**；seven-class Score(82–84) 事件流行度加权（1425/1429 落在 3 个可预测类），诚实数字是 macro-F1 0.36–0.39（4/7 类 F1=0）。
- **H-2 三个 frozen encoder 非受控对照**：backbone(AST/BEATs)、pooling(CLS vs 帧均值)、预处理(8s fbank vs 5s 裸波形)全不同，不能据此排名 Patch-Mix/PAFA/SG-SCL。
- **H-3 源锚点 author-test-selected**：`selection = author checkpoint selected on official test Score`（§1）；C0 与 clean-source anchor 均未做（index.csv 状态 package_ready / proposed）。无 C0 -> 任何 degradation 幅度 claim 不被授权（代码强制）。

**🟡 Medium**
- **M-1 "+28.18 pp"混用两种 Score 公式**（§5）。
- **M-2 "native 77.42"硬编码无内联引用**（run.py:756）；出处为 cross_dataset_comparisons.csv SPR-001 = SPRSound TBioCAS 2022 Table V，但那是 2022 CNN native baseline，对比 2024/25 微调 AST+linear head，架构/年代不对等，"+10.14 pp"须带 caveat。
- **M-3 PAFA/SG-SCL frozen-head 未注册、生成代码未提交**：experiments/index.csv 只有 patchmix 版；`baseline/common/frozen_encoder_target_heads.py`、`baseline/{pafa,sg_scl}/frozen_encoder_target_heads/` 处于 git 未跟踪。三分之二 headline 结果缺"已提交代码+注册实验"provenance。
- **M-4 内验证与 inter 类别构成不同 + best_epoch 恒=5**：val crackle-heavy、inter wheeze-heavy；三 head best_epoch 均=5（5 epoch 的最后一轮），validation "选择"不 binding，结果是"5-epoch 封顶"而非"validation-最优"。
- **M-5 过期证据指针**：metrics/source_alignment_receipt 与 docs 表 OURS-001 指向已迁移的 `result/patch_mix_cl_author_checkpoint_20260722_154852/`；OURS-002/004 仍 TBD。

**⚪ Low**：L-1 跨度量命名混淆（SPRSound 目标上的 (SE+SP)/2 也叫 icbhi_score）；L-2 narrow-four both support=1，任何 both 结论无意义（处理得当）；L-3 声学分布分析此前缺席（本次补齐）。

## Section 5 · Claim Ledger

**✅ Safe**：ICBHI 62.17 推理级精确复现；SPRSound Score 按官方 (AS+HS)/2；zero-target 源头直接迁移弱；协议卫生（冻结/丢弃/label 后加载/患者不相交/inter 只评一次）经 gate 与 receipt 核验。

**🟡 Conditional**：「frozen encoder + 轻量 head 大幅恢复 SPR 性能」——测得为真，但条件是 (a) 完整 target 监督非 zero-shot、(b) 无 AudioSet-only 对照 ICBHI 微调增量未证；「超过 native 77.42」——条件于架构/年代不对等 caveat；PAFA/SG-SCL 单 seed 接近论文——条件于源数为服务器接收。

**🔴 Unsupported（禁外宣）**：cross-dataset generalization / domain generalization；SOTA；"解决 imbalance"（macro-F1 0.36–0.39，4 类 F1=0）；三方法排名；degradation 幅度；"ICBHI 微调优于原始 AudioSet 表示"。

## Section 6 · Metric & Protocol Comparability

严格分栏、禁止跨栏相减：**ICBHI four-class Score**（cycle）｜**ICBHI binary Score**｜**SPRSound Task 1-1 AS/HS/Score**（event）｜**SPRSound Task 1-2 seven-class Score**（流行度加权，须并列 macro-F1）｜**narrow-four shared-ontology diagnostic**（诊断用，Rhonchi/Stridor 排除非并入 Wheeze，代码明确标注非官方 Task 1-2）。**红线**：❌ ICBHI 62 − SPRSound 90 ≠ 提升。合法对照只有同 target split 的 C0（未做）与 trivial floor（=50，已做）。

## Section 7 · Threats to Validity & Alternative Explanations（证伪）

| 假说 | 裁决 | 依据 |
|---|---|---|
| 高 SPR Score 只是 binary 更简单 | 部分成立 | binary 天然易；但同 encoder seven-class 也 82–84 |
| 主要来自完整 target 监督 | **成立（主因）** | zero-target 仅 +6–10pp |
| 由 inter 以 Wheeze 为主、稀有类缺失造成 | **成立** | 1425/1429 落在 3 类；官方 Score 事实退化为 3 类问题 |
| ICBHI 微调优于原始 AudioSet | **无法判定** | C-1，缺对照 |
| head 学到数据集/设备特征而非病理 | **无法排除** | 无 dataset-ID probe / 无声学分析（本次补上诊断） |
| 支持 SOTA/DG/解决 imbalance | 不支持 | §5 |
| patient/boundary/preproc 泄漏 | 主要路径已排除 | 患者不相交硬 gate；label-free manifest；annotation 边界；label 后加载。剩余 device/SR/duration shortcut 未分析（本次声学分析处理） |

**最可能的良性解释**：所有 SPR 强数值可由"AudioSet 强预训练 + 完整 SPR 监督 + 事件流行度加权度量"解释，无需诉诸"ICBHI 学到可迁移病理表示"。推翻它须 C-1 的 AudioSet-only 对照 + random-init 下界。

## Section 8 · Proposed Method Novelty Audit

**候选主线**：ontology-aware、imbalance-aware 多数据集系统（共享 encoder + coarse/fine heads + dataset-native heads + masked partial supervision + cross-head consistency + 双轴采样 + 可选 domain-conditioned adapter + human-reviewed versioned label overlay）。

| 组件 | 最近先例 | 新颖性 |
|---|---|---|
| 共享 encoder + dataset-native heads | 标准多任务 | 不新颖 |
| masked partial supervision | partial-label / 多数据集检测 | 应用新颖 |
| cross-head consistency（层级一致） | 层级分类；**且本项目 ICBHI joint-head/fusion 已为负结果** | 高风险（已在 ICBHI 上被自证伪） |
| domain-conditioned adapter | **BTS-CARD (ICASSP 2026)** 因果反事实+对抗+元数据/设备/位置去偏 | **直接冲突**，最弱一环 |
| 双轴采样 | 长尾采样标准 | 增量 |
| human-reviewed versioned label overlay | LungMix 语义标签插值 | dataset/resource 贡献，且与导师 06-29 张力 |

**三类贡献分离**：dataset/resource（label harmonization+eligibility mask，最可辩护但导师不希望作主线）；method（多为已知组件组合，相对 BTS-CARD+LungMix 新颖性不足）；**evaluation（真正空位：标准化 ontology-eligibility-masked、prevalence-controlled、leave-one-dataset-out 跨数据集呼吸声 benchmark）**。
**净评**：method-as-listed 不足以作独立方法贡献。差异化路线：(a) 把评估协议+benchmark 做成一等贡献；(b) 坐实"跨数据集失败主因是 label/接口 mismatch 而非声学域移"这一发现，再针对性设计接口对齐方法。

## Section 9 · Missing Experiments & Decision Gates

| # | 缺失实验 | 决策 gate | 优先级 |
|---|---|---|---|
| E1 | **AudioSet-only AST+BEATs + 同 head**（未 ICBHI 微调）on SPRSound | D1：若 ≈ ICBHI-FT → "ICBHI 表示可迁移"claim 死亡 | 🔴 最高 |
| E2 | random-init encoder + 同 head（下界） | encoder 贡献 vs head | 🔴 高 |
| E3 | C0 matched full-target fine-tune（已 package_ready） | head-only vs full-FT 差距 + target 可学性 | 🔴 高 |
| E4 | clean-source ICBHI anchor（不用 test 选）→ 重跑迁移 | 升级为可发表 source-only 证据 | 🟠 中 |
| E5 | leave-one-dataset-out（≥2 数据集双向） | 导师要求协议；DG claim 前提 | 🟠 中 |
| E6 | dataset-ID linear probe + 声学分布/设备分析 | 是否 shortcut/泄漏（**本次声学分析已启动**） | 🟠 中 |
| E7 | prevalence/class-balanced 评分子集 + per-class 表 | 击穿 seven-class 伪 3 类膨胀 | 🟡 中 |
| E8 | 最简 pooled / shared-encoder+dataset-heads baseline | 决定是否需 masked-loss/hierarchy/MoE | 🟡 中 |

## Section 10 · 最小两周执行计划（按依赖）

**Week 1（补致命对照，触发 D1）**：① E1 AudioSet-only 对照（复用 frozen-head pipeline，仅换 checkpoint）② E2 random-init 下界 ③ E3 C0 full-target（服务器）④ import & 独立复算 running_remote 五方法，修 M-3/M-5。→ **D1**：ICBHI-FT 是否显著 > AudioSet-only？head-only 是否逼近 C0？若否 → 放弃"共享表示可迁移"叙事。

**Week 2（定位失败机制，触发 D2）**：⑤ E4 clean-source anchor+重跑迁移 ⑥ E6 dataset-ID probe + 声学分布图（本次已完成第一版）⑦ E7 prevalence-controlled 评分 + E5 ICBHI↔SPRSound 双向 LODO ⑧ E8 最简 pooled baseline。→ **D2**：主导失败是 label 接口还是声学域？据此二选一，再决定是否投入 domain-adapter（否则撞 BTS-CARD）。

**依赖链**：E1/E2 是一切解释前提；E4 依赖 E3；E5/E8 依赖 E1 口径；主线选择依赖 D1+D2，不可提前锁定复杂结构（与 WORK_PLAN "New model entry: hold" 一致）。

## Section 11 · Go / No-Go

- 🟢 **GO** 继续诊断主线（source alignment → zero-target → frozen-head）。
- 🟢 **GO** 立即执行 E1/E2/E3（AudioSet-only、random-init、C0）——低成本、决定核心 claim。
- 🔴 **NO-GO** 现在锁定候选方法主线（ontology/consistency/domain-adapter）——对 BTS-CARD/LungMix 新颖性不成立，且 cross-head consistency 已在 ICBHI 被自证伪。
- 🔴 **NO-GO** 任何对外 claim：cross-dataset generalization / SOTA / solved imbalance / 三方法排名 / degradation 幅度。
- 🟡 **CONDITIONAL** 论文定位：把评估协议+benchmark 或"失败主因=标签接口"发现作为一等贡献，比再造 domain-adapter 更可辩护；绑定 D2。

---
### 外部事实核验（source_official_facts）
- **SPRSound 官方 Score**：IEEE BioCAS 2022 指标 = SE/SP/AS/HS 及"AS 与 HS 的平均(Score)"，与代码 `(AS+HS)/2` 一致（[官方 repo](https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound)）。
- **BTS-CARD**（ICASSP 2026，[arXiv 2510.22263](https://arxiv.org/abs/2510.22263)）：因果反事实 + 对抗去偏 + 元数据/设备/位置去偏，做 ICBHI↔SPRSound IND/OOD —— 与候选 domain-adapter 直接冲突。
- **LungMix**（[arXiv 2501.00064](https://arxiv.org/abs/2501.00064)）：ICBHI/SPR/HF 跨数据集，波形混合+语义标签插值，"可比肩 target-trained" —— 跨数据集泛化 + 标签语义先例。

*本审计只读核验；数字由本 notebook 代码从一手 `result/` 重算。*